# Wildlife Strike Analysis

*This notebook should be used to record all of your analysis.*

## Imports

In [126]:
import pandas as pd
import redshift_connector
import numpy as np

In [178]:
import redshift_connector
from dotenv import load_dotenv
from os import environ


def get_redshift_connection():
    """
    Create and return a Redshift connection using environment variables.
    Requires .env file with REDSHIFT_* variables.
    """
    load_dotenv()
    try:
        connection = redshift_connector.connect(
            host=environ.get("REDSHIFT_HOST"),
            database=environ.get("REDSHIFT_DB"),
            user=environ.get("REDSHIFT_USER"),
            password=environ.get("REDSHIFT_PASSWORD"),
            port=int(environ.get("REDSHIFT_PORT", "5439"))
        )
        print("Successfully connected to Redshift")
        return connection
    except Exception as e:
        print(f"Connection failed: {e}")
        raise


conn = get_redshift_connection()

try:
    with conn.cursor() as cursor:
        cursor.execute(
            "SELECT * FROM s_umelbaneen_al_mousawi.wildlife_strike")
        wildlife_df = cursor.fetch_dataframe()

        cursor.execute(
            "SELECT * FROM s_umelbaneen_al_mousawi.airline")
        airline_df = cursor.fetch_dataframe()

        cursor.execute(
            "SELECT * FROM s_umelbaneen_al_mousawi.airport")
        airport_df = cursor.fetch_dataframe()

        cursor.execute(
            "SELECT * FROM s_umelbaneen_al_mousawi.flight")
        flight_df = cursor.fetch_dataframe()

        cursor.execute(
            "SELECT * FROM s_umelbaneen_al_mousawi.laser_incident_placeholder")
        laser_incident_df = cursor.fetch_dataframe()

        cursor.execute(
            "SELECT * FROM s_umelbaneen_al_mousawi.city")
        city_df = cursor.fetch_dataframe()

        cursor.execute(
            "SELECT * FROM s_umelbaneen_al_mousawi.state")
        state_df = cursor.fetch_dataframe()




except Exception as e:
    print(f"Query failed: {e}")
finally:
    pass

Successfully connected to Redshift


### 1. How significant a problem are wildlife strikes?

#### Wildlife strikes are there per year

In [51]:
strike_counts = wildlife_df.groupby('incident_year').size().reset_index(name='strike_amount')
strike_counts['incident_year'] = pd.to_numeric(strike_counts['incident_year'])
strike_counts.dtypes

incident_year    int64
strike_amount    int64
dtype: object

#### Flights per year

In [42]:
flight_df['fl_date'] = pd.to_datetime(flight_df['fl_date'])

In [44]:
flight_df['incident_year'] = flight_df['fl_date'].dt.year

In [48]:
flight_count = flight_df.groupby('incident_year').size().reset_index(name='total_flights')
flight_count.dtypes

incident_year    int32
total_flights    int64
dtype: object

#### Comparison

In [ ]:

analysis_df = pd.merge(strike_counts, flight_count, on='incident_year')

In the flights column only the data for number of flights in 2023 exists

In [53]:
analysis_df.head()

,incident_year,strike_amount,total_flights
0,2023,19628,570394


In [ ]:
total_strikes = analysis_df['strike_amount']    
total_flights = analysis_df['total_flights']

0    19628
Name: strike_amount, dtype: int64

In [71]:
incident_rate = (total_strikes/total_flights) * 100
round(incident_rate, 2)

0    3.44
dtype: float64

In [76]:
number_of_flights_affected = 570394*0.0344
number_of_flights_affected

19621.5536

wildlife strikes affected 3.44% of total flights in 2023. 19,621 flights where affected in 2023

### 2. Are strikes by particular animals more likely/dangerous than others?

#### Most Frequent Animal Striker

In [82]:
impact_cols = [
    'species',
    'nr_injuries',
    'nr_fatalities',
    'indicated_damage',
    'damage_level',
    'cost_repairs'
]


In [83]:
wildlife_df[impact_cols].head()

,species,nr_injuries,nr_fatalities,indicated_damage,damage_level,cost_repairs
0,Unknown bird - medium,,,0,N,
1,Unknown bird - medium,,,0,N,
2,Unknown bird - small,,,0,N,
3,Ducks,,,1,M,
4,Mourning dove,,,1,M?,


In [86]:
most_likely = wildlife_df.groupby(
    'species').size().reset_index(name='strike_count')

In [87]:
most_likely = most_likely.sort_values(by='strike_count', ascending=False)

print("Top 5 Most Likely Species to be Hit:")
print(most_likely.head(5))

Top 5 Most Likely Species to be Hit:
                   species  strike_count
801   Unknown bird - small         50726
800  Unknown bird - medium         38709
798           Unknown bird         27860
515          Mourning dove         15331
52            Barn swallow         10083


#### Most Dangerous

In [88]:
damage_map = {'D': 10, 'S': 5, 'M': 1, 'M?': 1, 'N': 0}

In [91]:
wildlife_df['severity_score'] = wildlife_df['damage_level'].map(
    damage_map).fillna(0)



In [93]:
most_dangerous = wildlife_df.groupby('species').agg(
    strike_count=('species', 'count'),
    avg_severity=('severity_score', 'mean'),
    total_severity=('severity_score', 'sum')
).reset_index()
most_dangerous

,species,strike_count,avg_severity,total_severity
0,Acadian flycatcher,21,0.000000,0.0
1,African collared dove,2,0.000000,0.0
2,African silverbill,5,0.000000,0.0
3,African yellow bat,1,0.000000,0.0
4,Alder flycatcher,24,0.000000,0.0
...,...,...,...,...
911,Yellow-throated vireo,9,0.000000,0.0
912,Yellow-throated warbler,50,0.020000,1.0
913,Yuma myotis,3,0.000000,0.0
914,Zebra dove,488,0.018443,9.0


In [94]:
top_danger = most_dangerous[most_dangerous['strike_count'] > 5].sort_values(
    by='avg_severity', ascending=False)

print("Top 5 Most Dangerous Species (By Average Severity):")
print(top_danger.head(5))

Top 5 Most Dangerous Species (By Average Severity):
          species  strike_count  avg_severity  total_severity
167        Cattle            12      5.583333            67.0
817  Wapiti (elk)            12      4.416667            53.0
509         Moose             6      3.500000            21.0
792   Tundra swan            26      2.538462            66.0
193  Common eider             6      2.500000            15.0


In [95]:
import altair as alt

chart = alt.Chart(top_danger).mark_circle(size=100).encode(
    x=alt.X('strike_count:Q', title='Total Strikes (Likelihood)'),
    y=alt.Y('avg_severity:Q', title='Average Severity Score (Danger)'),
    color=alt.Color('avg_severity:Q', scale=alt.Scale(scheme='reds')),
    tooltip=['species', 'strike_count', 'avg_severity']
).properties(
    title='Wildlife Strikes: Likelihood vs. Danger',
    width=600,
    height=400
)

chart.display()

alt.Chart(...)

In 2023, wildlife strikes proved to be a significant operational challenge, affecting approximately 3.44% of all flights, which equates to roughly 1 in every 29 departures. While the most frequent incidents involved small or medium-sized birds—often resulting in negligible damage—the data reveals a sharp contrast between likelihood and severity. Ground-based animals, such as cattle and elk, represented the most dangerous threats; despite their rarity, these collisions yielded the highest average severity scores due to the substantial damage caused to aircraft. Consequently, while avian encounters are a daily nuisance for the aviation industry, it is the infrequent strikes with larger mammals that pose the greatest risk to flight safety and structural integrity.

## 3. When and in what conditions are strikes most likely?

Total strikes per month

In [106]:
wildlife_df['incident_month'] = pd.to_numeric(wildlife_df['incident_month'])

In [107]:
monthly_strikes = wildlife_df.groupby(
    'incident_month').size().reset_index(name='strike_count').sort_values('incident_month')

monthly_strikes

,incident_month,strike_count
0,1,10125
1,2,9732
2,3,14991
3,4,22358
4,5,31016
5,6,26148
6,7,37764
7,8,41885
8,9,40124
9,10,37745


In [117]:
wildlife_df = wildlife_df.dropna(how='all')

In [133]:
wildlife_df['time_of_day'] = wildlife_df['time_of_day'].replace('', np.nan)

In [128]:
time_of_day_strikes = wildlife_df.groupby('time_of_day').size().reset_index(name='strike_amount').sort_values('strike_amount', ascending=False)
time_of_day_strikes

,time_of_day,strike_amount
1,Day,105476
3,Night,53203
2,Dusk,7669
0,Dawn,6059


In [137]:
wildlife_df['weather_condition'] = wildlife_df['weather_condition'].replace(
    '', np.nan)

In [146]:
wildlife_df['weather_condition'] = wildlife_df['sky'].astype(
    str) + " / " + wildlife_df['precipitation'].fillna('None').astype(str)

In [147]:
wildlife_df['weather_condition'] = wildlife_df['weather_condition'].replace(
    [' / ', 'nan / None', 'None / None'], 'Unknown / Not Reported')

In [148]:
weather_strikes = wildlife_df.groupby('weather_condition').size().reset_index(
    name='strike_count').sort_values('strike_count', ascending=False)

In [149]:
weather_strikes.head()

,weather_condition,strike_count
25,Unknown / Not Reported,158368
4,No Cloud /,70332
19,Some Cloud /,48257
10,Overcast /,15704
16,Overcast / Rain,6538


In [151]:
weather_strikes_filtered = weather_strikes[weather_strikes['weather_condition']
                                           != 'Unknown / Not Reported']

weather_strikes_filtered.head()

,weather_condition,strike_count
4,No Cloud /,70332
19,Some Cloud /,48257
10,Overcast /,15704
16,Overcast / Rain,6538
11,Overcast / Fog,1830


In [153]:
# 1. Seasonal Analysis: Strikes by Month
# Based on image_d82084.png
month_chart = alt.Chart(monthly_strikes).mark_bar(color='#5276A7').encode(
    x=alt.X('incident_month:O', title='Month (Jan-Dec)'),
    y=alt.Y('strike_count:Q', title='Total Strikes'),
    tooltip=['incident_month', 'strike_count']
).properties(
    title='Strike Frequency by Month',
    width=400,
    height=300
)

month_chart.display()

alt.Chart(...)

In [ ]:

tod_chart = alt.Chart(time_of_day_strikes).mark_arc(innerRadius=50).encode(
    theta=alt.Theta(field="strike_amount", type="quantitative"),
    color=alt.Color(field="time_of_day", type="nominal", title="Time of Day"),
    tooltip=['time_of_day', 'strike_amount']
).properties(
    title='Strikes by Time of Day',
    width=300,
    height=300
)

tod_chart.display()

alt.Chart(...)

In [157]:
weather_chart = alt.Chart(weather_strikes_filtered.head(10)).mark_bar(color='#F18727').encode(
    x=alt.X('strike_count:Q', title='Total Strikes'),
    y=alt.Y('weather_condition:N', sort='-x',
            title='Condition (Sky / Precip)'),
    tooltip=['weather_condition', 'strike_count']
).properties(
    title='Top 10 Reported Weather Conditions',
    width=400,
    height=300
)

weather_chart.display()

alt.Chart(...)

The "Migration" Peak: Your data shows a massive climb starting in July (37,764) and peaking in August (41,885). This aligns with the seasonal period when young birds take flight and fall migration begins.

Daylight Risk: Strikes are significantly more likely during the Day (105,476) compared to Night (53,203). This represents a daylight strike rate of approximately 63%.

Clear Skies: Once the unknowns are removed, the "No Cloud /" condition is the most common environment for strikes (70,332). This suggests that visibility for the pilot doesn't necessarily prevent strikes, as high flight volume occurs during clear weather.

## 4. Which airlines/airports/states would be likely potential customers for any of this technology?

In [190]:

airport_counts = wildlife_df.groupby(
    'airport').size().reset_index(name='strike_count')

airport_counts = airport_counts[~airport_counts['airport'].isin(
    ['UNKNOWN', 'OFF AIRPORT'])]

top_airport = airport_counts.sort_values(
    'strike_count', ascending=False).iloc[0]

print(
    f"Top Airport Customer: {top_airport['airport']} ({top_airport['strike_count']} strikes)")

Top Airport Customer: DENVER INTL AIRPORT (10057 strikes)


In [185]:
wildlife_df['index_nr'] = pd.to_numeric(wildlife_df['index_nr'])

In [192]:
state_leads = wildlife_df.groupby(
    'enroute_state').size().reset_index(name='strike_count')

state_leads = state_leads[~state_leads['enroute_state'].isin(
    ['UNKNOWN', '', 'Unknown', 'None'])]

top_state = state_leads.sort_values('strike_count', ascending=False).iloc[0]

print(
    f"Target State Customer: {top_state['enroute_state']} ({top_state['strike_count']} strikes)")

Target State Customer: FL (556 strikes)


Strikes happen most frequently in florida.